# 第8章 风险度量与压力测试

> **核心问题**：风险能否被一个数字概括？波动率、回撤、VaR和Expected Shortfall分别看见什么，又遗漏什么？

- 金融线：波动、下行、回撤、尾部损失、压力情景和杠杆。
- 数学线：标准差、半方差、路径极值、分位数和条件均值。
- Python线：滚动窗口、自定义风险函数、历史模拟、图层与边界测试。

## AI学习状态

当前进度：第8章开始  
已掌握：收益分布、分位数和抽样  
仍然薄弱：待填写  
下一步：每个风险指标都写出对象、期限和置信水平。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"]=(8,4.5); plt.rcParams["axes.grid"]=True
plt.rcParams["font.sans-serif"]=["Arial Unicode MS","PingFang SC","SimHei","DejaVu Sans"]
plt.rcParams["axes.unicode_minus"]=False
rng=np.random.default_rng(20260711)

## 8.1 波动率：围绕平均值的离散程度

样本标准差衡量收益围绕样本均值的变化。年化日波动率常用`日标准差×sqrt(252)`，但它依赖独立、稳定和交易日数等假设，不能机械套用。

In [ ]:
dates=pd.date_range("2024-01-01",periods=500,freq="B")
returns=pd.Series(rng.standard_t(5,500)*.012,index=dates,name="return")
daily_vol=returns.std(ddof=1); annual_vol=daily_vol*np.sqrt(252)
downside=np.sqrt(np.mean(np.minimum(returns,0)**2))*np.sqrt(252)
print({"日波动率":f"{daily_vol:.2%}","年化波动率":f"{annual_vol:.2%}","年化下行偏差":f"{downside:.2%}"})

### 思考

同样大小的上涨和下跌都会提高波动率，但投资者感受是否相同？下行偏差为什么仍不能涵盖流动性或信用风险？

### 我的回答

<!-- 在这里填写；完成前AI不要代答 -->

## 8.2 回撤是路径指标

财富 $W_t$ 相对历史峰值 $M_t=\max_{s\le t}W_s$ 的回撤：

$$DD_t=\frac{W_t}{M_t}-1$$

最大回撤是样本期内最小的$DD_t$。它依赖观察区间和路径，不能告诉我们未来最大损失。

In [ ]:
wealth=10_000*(1+returns).cumprod()
running_peak=wealth.cummax()
drawdown=wealth/running_peak-1
max_dd=drawdown.min(); trough=drawdown.idxmin(); peak=wealth.loc[:trough].idxmax()
print({"最大回撤":f"{max_dd:.2%}","峰值日期":str(peak.date()),"谷底日期":str(trough.date())})

fig,axes=plt.subplots(2,1,figsize=(9,7),sharex=True)
wealth.plot(ax=axes[0],title="财富与历史峰值"); running_peak.plot(ax=axes[0],ls="--",label="历史峰值"); axes[0].legend()
drawdown.plot(ax=axes[1],color="crimson",title="回撤"); axes[1].fill_between(drawdown.index,drawdown,0,alpha=.25,color="crimson")
plt.tight_layout(); plt.show()

## 8.3 VaR：一个损失分位点

对收益分布，95%的一日VaR可写为`-收益的5%分位数`。它表示在模型/样本下，有约5%的日收益比该阈值更差；它不描述超过阈值后会多糟。

In [ ]:
alpha=.05
var95=-returns.quantile(alpha)
tail=returns[returns<=returns.quantile(alpha)]
es95=-tail.mean()
print({"95%一日VaR":f"{var95:.2%}","95%一日Expected Shortfall":f"{es95:.2%}","尾部样本数":len(tail)})

Expected Shortfall（ES）是落入最差$\alpha$尾部时损失的平均，能反映阈值以外的严重程度，但同样依赖样本、模型、期限和估计方法。

In [ ]:
q=returns.quantile(.05)
fig,ax=plt.subplots(); ax.hist(returns,bins=60,density=True,alpha=.7)
ax.axvline(q,color="red",label=f"5%分位={q:.2%}"); ax.axvspan(returns.min(),q,color="red",alpha=.2,label="ES对应尾部")
ax.set(title="历史收益分布、VaR阈值与尾部",xlabel="日收益",ylabel="密度"); ax.legend(); plt.show()

### 我的解释

“95% VaR为2%”为什么不能说“最大只会亏2%”？如果样本从未经历危机，历史VaR会有什么问题？

<!-- 在这里填写；完成前AI不要代答 -->

## 8.4 正态VaR与历史VaR可能给出不同答案

正态法用均值和标准差概括分布，历史法直接使用经验分位。厚尾、偏度和状态变化会让结果明显不同。

In [ ]:
from scipy.stats import norm
mu,sigma=returns.mean(),returns.std(ddof=1)
normal_var=-(mu+sigma*norm.ppf(.05))
historical_var=-returns.quantile(.05)
print({"正态VaR":f"{normal_var:.2%}","历史VaR":f"{historical_var:.2%}"})

## 8.5 滚动风险揭示非稳定性

整段样本的一个波动率会掩盖不同市场状态。滚动估计更接近“当时可知”，但窗口选择也会影响灵敏度和噪声。

In [ ]:
regime=np.r_[rng.normal(0,.006,250),rng.normal(0,.025,250)]
regime=pd.Series(regime,index=dates)
rolling_vol=regime.rolling(60).std()*np.sqrt(252)
regime.cumsum().plot(label="累计简单和（仅观察）"); plt.twinx(); rolling_vol.plot(color="red",label="60日年化波动")
plt.title("市场状态变化与滚动波动率"); plt.show()

## 8.6 压力测试：主动提出历史之外的问题

压力测试直接施加情景，如股票-30%、债券-8%、现金0%，研究组合结果。情景不是概率预测，而是脆弱性检查。

In [ ]:
weights=pd.Series({"股票":.6,"债券":.3,"现金":.1})
scenarios=pd.DataFrame({
    "温和下跌":{"股票":-.10,"债券":.02,"现金":.00},
    "股债同跌":{"股票":-.30,"债券":-.08,"现金":.00},
    "快速反弹":{"股票":.20,"债券":-.03,"现金":.00},
}).T
scenarios["组合收益"]=scenarios.mul(weights,axis=1).sum(axis=1)
display(scenarios.style.format("{:.1%}"))

## 8.7 杠杆放大收益，也放大生存风险

简单教学模型中，杠杆$L$把风险资产收益放大，并扣融资成本。一次-50%收益在2倍杠杆下可能耗尽资本，之后无法靠普通百分比反弹恢复。

In [ ]:
losses=np.linspace(0,-.60,121)
for leverage in [1,1.5,2,3]:
    equity_return=leverage*losses
    plt.plot(losses,equity_return,label=f"{leverage}倍")
plt.axhline(-1,color="black",ls="--",label="资本耗尽")
plt.xlabel("资产收益"); plt.ylabel("简化权益收益"); plt.title("杠杆与资本损失（未计融资和强平细节）"); plt.legend(); plt.show()

**量化编程警告**：现实中保证金、逐日盯市、融资成本、跳空和强平规则会使杠杆路径更复杂，不能用简单乘法替代真实风险管理。

## 8.8 编程练习：风险摘要

返回波动率、最大回撤、历史VaR和ES；检查置信水平在0与1之间且收益大于-100%。

In [ ]:
def risk_summary(returns,alpha=.05):
    # TODO
    return None

In [ ]:
ans=risk_summary([.10,-.10,.05,-.20],.25)
if ans is None: print("练习尚未完成。")
else: print(ans)

## 本章总结与小项目

比较两个“平均收益相同”的策略：报告年化波动、下行偏差、最大回撤、历史/正态VaR、ES和三个压力情景；解释每个指标遗漏的风险。

**底线**：风险是多维的；指标是观察窗口，不是安全证明。